# Problemas

Cada aluno receberá um dos quatro problemas abaixo e terá **uma semana** para resolvê-lo, em um notebook próprio no Colab.

**Entrega:** o notebook deve ser enviado pelo Classroom **até o início da aula seguinte**, organizado em seções (use a numeração dos itens), com os enunciados, as explicações em células Markdown e as fórmulas em LaTeX.

**Apresentação oral (cerca de 15 minutos):**

1. **Formulação:** apresente o modelo, o significado das matrizes e vetores envolvidos e as hipóteses.
2. **Resultados:** mostre os gráficos e números principais e explique o que eles dizem sobre o problema.
3. **Investigação:** apresente sua resposta para as perguntas abertas do problema.

Não leia o código linha por linha: explique a ideia de cada função e esteja preparado para responder perguntas sobre qualquer parte do notebook, inclusive alterando parâmetros ao vivo.

**Regras gerais:** todo gráfico deve ter título, rótulos nos eixos e legenda quando houver mais de uma curva. O uso de IA é permitido, mas você deve ser capaz de justificar cada escolha feita no notebook.

### **Problema 1: O PageRank do Google**

No final dos anos 1990, Larry Page e Sergey Brin propuseram uma forma de medir a importância de uma página da web que deu origem ao Google: **uma página é importante se recebe links de outras páginas importantes**. Essa definição circular é resolvida com álgebra linear.

Imagine um "navegante aleatório" que, a cada clique, escolhe ao acaso um dos links da página em que está. Para uma web com $n$ páginas, seja $H$ a matriz $n \times n$ com

$$
H_{ij} = \begin{cases} 1/L_j, & \text{se a página } j \text{ tem um link para a página } i,\\ 0, & \text{caso contrário,}\end{cases}
$$

onde $L_j$ é o número de links que saem da página $j$. Se uma página não tem links de saída (um "beco sem saída"), supomos que o navegante pula para uma página qualquer, ou seja, a coluna correspondente de $H$ tem todas as entradas iguais a $1/n$.

Para evitar que o navegante fique preso em grupos de páginas, a cada clique ele segue um link com probabilidade $\alpha$ e, com probabilidade $1 - \alpha$, digita o endereço de uma página qualquer. Isso leva à **matriz do Google**

$$
G = \alpha H + \frac{1-\alpha}{n}\,\mathbf{1}\mathbf{1}^T,
$$

onde $\mathbf{1}\mathbf{1}^T$ é a matriz $n \times n$ de uns. O Google usava $\alpha = 0.85$. O **PageRank** é o vetor $\mathbf{p}$ com entradas positivas, somando 1, tal que $G\mathbf{p} = \mathbf{p}$: a entrada $p_i$ é a fração do tempo que o navegante passa, a longo prazo, na página $i$.

Considere a seguinte web com 8 páginas, em que cada página aponta para as páginas indicadas:

| Página | A | B | C | D | E | F | G | H |
|---|---|---|---|---|---|---|---|---|
| Links para | B, C | C, E | A | C, G | C, F, H | (nenhum) | D, H | E, G |

**Tarefas:**

1. **Formulação.** Explique por que as colunas de $H$ e de $G$ somam 1. Usando esse fato, mostre que $\lambda = 1$ é autovalor de $G$ (dica: olhe para $G^T\mathbf{1}$). Explique, em termos do navegante aleatório, por que $G\mathbf{p} = \mathbf{p}$ define uma distribuição de equilíbrio.

2. **Montagem.** Escreva uma função `matriz_H(links, paginas)` que receba um dicionário de links (como `{'A': ['B','C'], ...}`) e a lista de páginas e retorne a matriz $H$, tratando corretamente as páginas sem links. Monte $H$ e $G$ (com $\alpha = 0.85$) para a web acima e desenhe o grafo direcionado com `networkx` (`nx.DiGraph`).

3. **PageRank pelos autovetores.** Use `la.eig` para encontrar o autovetor de $G$ associado a $\lambda = 1$, normalize-o para que some 1 e apresente o ranking das páginas em uma tabela. Compare com `nx.pagerank`. A página que recebe mais links é a mais importante? Explique.

4. **Método das potências.** Implemente o método das potências $\mathbf{p}_{k+1} = G\mathbf{p}_k$, a partir de $\mathbf{p}_0 = \frac{1}{n}\mathbf{1}$, parando quando $\|\mathbf{p}_{k+1} - \mathbf{p}_k\|_1 < 10^{-10}$.
    * Por que aqui não é necessário normalizar o vetor a cada passo?
    * Plote o erro $\|\mathbf{p}_{k+1} - \mathbf{p}_k\|_1$ em função de $k$ em escala semilogarítmica e compare a inclinação com $|\lambda_2|$, o segundo maior módulo entre os autovalores de $G$.
    * Verifique numericamente que $|\lambda_2(G)| = \alpha\,|\lambda_2(H)|$.

5. **O papel de $\alpha$.** Para $\alpha = 0.5, 0.7, 0.85, 0.9, 0.95, 0.99$, calcule o número de iterações do método das potências e o ranking das páginas. Apresente os resultados em uma tabela e em um gráfico. O que acontece com o ranking e com a velocidade de convergência quando $\alpha$ se aproxima de 1? E quando $\alpha$ se aproxima de 0? Por que o Google não usava $\alpha$ muito próximo de 1?

6. **Investigação: manipulando o ranking.** O dono da página D, que está entre as últimas do ranking, quer chegar ao primeiro lugar criando páginas falsas (uma "fazenda de links").
    * **Estratégia 1:** criar $k$ páginas novas, cada uma com um único link para D (D mantém seus links). Calcule a posição de D no ranking das 8 páginas originais para $k = 0, 1, \dots, 10$.
    * **Estratégia 2:** criar $k$ páginas novas com links para D e, além disso, D passa a ter links **apenas** para essas páginas novas. Repita o cálculo.
    * Compare as duas estratégias e explique o resultado em termos do navegante aleatório. Que característica do grafo a estratégia 2 explora? Que tipo de medida um buscador poderia adotar para se proteger?

### **Problema 2: Temperatura em uma Placa**

Uma placa metálica quadrada, de lado 1, tem as bordas mantidas a temperaturas fixas: a borda superior a 100 °C e as outras três a 0 °C. Depois de muito tempo, a temperatura $T(x,y)$ na placa se estabiliza e satisfaz a **equação de Laplace** $T_{xx} + T_{yy} = 0$.

Para resolvê-la numericamente, dividimos a placa em uma malha com $N \times N$ pontos interiores, espaçados de $h = 1/(N+1)$: o ponto $(i,j)$, com $i, j = 0, \dots, N-1$, fica em $(x_j, y_i) = ((j+1)h,\ (i+1)h)$, de modo que $i = 0$ é a linha mais próxima da borda inferior. Aproximando as derivadas segundas por diferenças centrais, a equação de Laplace se torna

$$
T_{i,j} = \frac{T_{i+1,j} + T_{i-1,j} + T_{i,j+1} + T_{i,j-1}}{4},
$$

ou seja, **a temperatura em cada ponto é a média das temperaturas dos quatro vizinhos**. Quando um vizinho está na borda, usamos a temperatura conhecida da borda. Numerando as incógnitas por $k = iN + j$, obtemos um sistema linear $A\mathbf{T} = \mathbf{b}$ com $n = N^2$ incógnitas.

**Tarefas:**

1. **Formulação.**
    * Mostre que a fórmula da média vem de aproximar $T_{xx}$ e $T_{yy}$ pela diferença central $f''(x) \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2}$.
    * Escreva explicitamente (em LaTeX) a matriz $A$ e o vetor $\mathbf{b}$ para $N = 3$. Qual é a estrutura de $A$? Quantas entradas não nulas tem cada linha, no máximo?

2. **Montagem e solução.** Escreva uma função `monta_sistema(N, T_topo, T_base, T_esq, T_dir)` que retorne $A$ e $\mathbf{b}$. Resolva o sistema para $N = 40$ com `la.solve` e visualize a temperatura com `plt.contourf` (incluindo as bordas), desenhando também algumas isotermas com `plt.contour`.

3. **Validação.**
    * A solução exata deste problema é a série
    $$T(x,y) = \sum_{n \text{ ímpar}} \frac{400}{n\pi}\,\frac{\sinh(n\pi y)}{\sinh(n\pi)}\,\sin(n\pi x).$$
    Calcule $T(0.5, 0.75)$ pela série (somando termos suficientes) e compare com a solução numérica para $N = 3, 7, 15, 31, 63$ (para esses valores, o ponto $(0.5, 0.75)$ pertence à malha). Faça o gráfico log-log do erro em função de $h$ e estime a ordem do método.
    * Usando o **princípio da superposição** (a soma de soluções é solução, para as somas das condições de borda) e a simetria do quadrado, mostre que a temperatura no centro da placa é exatamente 25 °C. Confira numericamente para $N$ ímpar.

4. **Custo e esparsidade.**
    * Que fração das entradas de $A$ é não nula? Como essa fração se comporta quando $N$ cresce?
    * Meça o tempo para resolver o sistema com `la.solve` (matriz densa) para $N = 10, 20, 30, 40, 50, 60$, e com `scipy.sparse.linalg.spsolve`, usando $A$ no formato esparso `scipy.sparse.csr_matrix`, para $N$ até 200. Faça um gráfico log-log dos tempos em função do número de incógnitas $n = N^2$ e estime os expoentes. Quanta memória ocuparia a matriz densa para $N = 200$?

5. **Método iterativo de Jacobi.** Em vez de montar a matriz, podemos partir de um palpite (por exemplo, $T = 0$ no interior) e substituir repetidamente cada valor pela média dos vizinhos.
    * Implemente o método de Jacobi usando fatias de arrays NumPy (sem laços sobre os pontos da malha), parando quando a maior mudança entre duas iterações for menor que $10^{-6}$. Conte as iterações para $N = 10, 20, 40$.
    * Em forma matricial, o método é $\mathbf{T}_{k+1} = J\mathbf{T}_k + \mathbf{b}/4$, com $J = I - A/4$. O erro diminui, a cada iteração, aproximadamente pelo fator $\rho(J)$, o **raio espectral** de $J$ (o maior módulo de seus autovalores). Calcule $\rho(J)$ com `la.eigvals` para $N = 10$ e compare com a fórmula teórica $\rho(J) = \cos\left(\frac{\pi}{N+1}\right)$.
    * Use $\rho(J)$ para estimar o número de iterações necessárias e compare com o observado. Por que o número de iterações cresce aproximadamente como $N^2$?

6. **Investigação.**
    * **Princípio do máximo:** verifique numericamente que as temperaturas máxima e mínima no interior estão entre as temperaturas das bordas. Usando a fórmula da média, prove que nenhum ponto interior pode ser mais quente que todos os seus vizinhos.
    * **Projeto:** mantendo a borda superior a 100 °C e a esquerda e a inferior a 0 °C, a que temperatura deve ser mantida a borda direita para que o ponto $(0.75,\ 0.25)$ fique a 30 °C? Resolva usando a linearidade do problema, com apenas duas soluções do sistema (e sem tentativa e erro).

### **Problema 3: A Órbita de um Cometa e o Método de Gauss**

Em 1º de janeiro de 1801, o astrônomo Giuseppe Piazzi descobriu Ceres, o primeiro asteroide conhecido, mas conseguiu observá-lo por apenas 41 dias antes que ele se perdesse no brilho do Sol. Os astrônomos não sabiam onde procurá-lo novamente até que Carl Friedrich Gauss, então com 24 anos, calculou sua órbita a partir daquelas poucas observações. Ceres foi reencontrado exatamente onde Gauss previu. Uma das ferramentas que ele desenvolveu nesse trabalho foi o **método dos mínimos quadrados**.

Pela Primeira Lei de Kepler, a órbita de um corpo em torno do Sol é uma cônica (uma elipse, no caso de órbitas fechadas) com o Sol em um dos focos. A tabela abaixo contém 12 observações (com erros de medição) da posição de um cometa no plano de sua órbita, em unidades astronômicas (UA), com o Sol na origem:

```python
x = np.array([ 1.069,  1.320,  1.388,  1.300,  1.065,  0.755,  0.383, -0.053, -0.668, -1.533, -2.478, -3.531])
y = np.array([-1.859, -1.056, -0.437,  0.173,  0.571,  0.896,  1.181,  1.379,  1.489,  1.409,  0.974, -0.020])
```

**Tarefas:**

1. **Formulação.** Toda cônica pode ser escrita como $Ax^2 + Bxy + Cy^2 + Dx + Ey + F = 0$. Explique por que, se a cônica não passa pela origem, podemos tomar $F = -1$, obtendo

    $$Ax^2 + Bxy + Cy^2 + Dx + Ey = 1.$$

    Escreva o sistema linear $M\mathbf{c} = \mathbf{1}$ (com $\mathbf{c} = [A,B,C,D,E]^T$) obtido ao impor que os 12 pontos estejam na cônica. Por que esse sistema, em geral, não tem solução? O que significa resolvê-lo "no sentido dos mínimos quadrados"?

2. **Ajuste.** Encontre $\mathbf{c}$ resolvendo as equações normais $M^TM\mathbf{c} = M^T\mathbf{1}$ e também com `la.lstsq`. Compare os resultados, a soma dos quadrados dos resíduos e os números de condição de $M$ e de $M^TM$. Que relação existe entre eles?

3. **Visualização.** Plote as observações, a cônica ajustada e o Sol. Para desenhar a cônica, calcule $f(x,y) = Ax^2 + Bxy + Cy^2 + Dx + Ey - 1$ em uma malha (`np.meshgrid`) e use `plt.contour(X, Y, f, levels=[0])`. Use `plt.axis('equal')`.

4. **Geometria pelos autovalores.** Escrevendo $\mathbf{x} = [x,y]^T$, a equação da cônica é $\mathbf{x}^TQ\mathbf{x} + \mathbf{d}^T\mathbf{x} = 1$, com $Q = \begin{bmatrix} A & B/2 \\\ B/2 & C \end{bmatrix}$ e $\mathbf{d} = [D,E]^T$.
    * Mostre que o centro da cônica é $\mathbf{x}_0 = -\frac{1}{2}Q^{-1}\mathbf{d}$ e que, em relação ao centro, a equação fica $(\mathbf{x}-\mathbf{x}_0)^TQ(\mathbf{x}-\mathbf{x}_0) = k$, com $k = 1 + \mathbf{x}_0^TQ\mathbf{x}_0$.
    * Usando a diagonalização de $Q$ (com `la.eigh`), explique por que a cônica é uma elipse quando os dois autovalores têm o mesmo sinal que $k$, e por que seus semieixos são $\sqrt{k/\lambda_1}$ e $\sqrt{k/\lambda_2}$ nas direções dos autovetores.
    * Calcule o centro, os semieixos $a$ e $b$, a excentricidade $e = \sqrt{1 - b^2/a^2}$, a inclinação do eixo maior e os dois focos ($\mathbf{x}_0 \pm ae\,\mathbf{u}$, onde $\mathbf{u}$ é o autovetor unitário do eixo maior). O Sol está (aproximadamente) em um dos focos?
    * Calcule as distâncias do periélio e do afélio e, pela Terceira Lei de Kepler ($T^2 = a^3$, com $T$ em anos e $a$ em UA), o período do cometa.

5. **Usando a física: o modelo focal.** Sabendo que o Sol está em um foco, a órbita em coordenadas polares é $r = \dfrac{p}{1 + e\cos(\theta - \omega)}$, onde $p$ é o *semilatus rectum* e $\omega$ indica a direção do periélio.
    * Mostre que essa equação pode ser escrita como $r = p - u\,x - v\,y$, com $u = e\cos\omega$ e $v = e\sin\omega$, que é **linear** nos parâmetros $p$, $u$, $v$.
    * Ajuste esse modelo às observações por mínimos quadrados e obtenha $e$, $\omega$ e $a = p/(1 - e^2)$. Compare com os resultados do item 4 e desenhe as duas órbitas no mesmo gráfico.

6. **Investigação: quanto da órbita precisamos ver?** Tome a órbita obtida no item 5 como "verdadeira" e escreva uma função que gere $m = 12$ observações sintéticas, igualmente espaçadas em ângulo ao longo de um arco de $\Delta\theta$ graus, com ruído normal de desvio-padrão $\sigma = 0.03$ UA em cada coordenada.
    * Para $\Delta\theta = 300°, 180°, 90°, 45°$, repita o experimento 300 vezes, ajustando os dois modelos (cônica geral e modelo focal) e guardando o semieixo maior $a$ estimado. Faça histogramas ou *boxplots* e uma tabela com a mediana e a dispersão de $a$ em cada caso.
    * Em quais situações o modelo da cônica geral falha (por exemplo, produzindo uma hipérbole)? Por que o modelo focal é mais robusto? Relacione com a dificuldade do problema enfrentado por Gauss.

### **Problema 4: Como Salvar as Tartarugas Marinhas?**

Nos anos 1980, as populações de tartaruga-cabeçuda (*Caretta caretta*), a mesma espécie que desova no litoral do Brasil, estavam em declínio nos Estados Unidos, e a principal estratégia de conservação era proteger os ovos e os filhotes nas praias. Em um estudo que se tornou clássico (Crouse, Crowder e Caswell, *Ecology*, 1987), os pesquisadores usaram um modelo matricial para perguntar: **em qual fase da vida a proteção tem mais efeito sobre a população?**

A população foi dividida em 7 estágios: (1) ovos e filhotes, (2) juvenis pequenos, (3) juvenis grandes, (4) subadultos, (5) reprodutores novatos, (6) reprodutores do 1º ano de retorno e (7) reprodutores maduros. Se $\mathbf{x}_k \in \mathbb{R}^7$ contém o número de fêmeas em cada estágio no ano $k$, então $\mathbf{x}_{k+1} = L\mathbf{x}_k$, com

$$
L = \begin{bmatrix}
0 & 0 & 0 & 0 & 127 & 4 & 80 \\\
0.6747 & 0.7370 & 0 & 0 & 0 & 0 & 0 \\\
0 & 0.0486 & 0.6610 & 0 & 0 & 0 & 0 \\\
0 & 0 & 0.0147 & 0.6907 & 0 & 0 & 0 \\\
0 & 0 & 0 & 0.0518 & 0 & 0 & 0 \\\
0 & 0 & 0 & 0 & 0.8091 & 0 & 0 \\\
0 & 0 & 0 & 0 & 0 & 0.8091 & 0.8089
\end{bmatrix}.
$$

**Tarefas:**

1. **Formulação.** Interprete as entradas de $L$: o que representam a primeira linha, a diagonal e a subdiagonal? Por que só os estágios 5, 6 e 7 aparecem na primeira linha? Qual é a probabilidade de uma tartaruga do estágio 3 sobreviver por um ano (permanecendo no estágio ou avançando)? Desenhe o ciclo de vida como um grafo direcionado (com `networkx`, se quiser).

2. **Simulação.** A partir de $\mathbf{x}_0 = [2000, 3000, 1000, 300, 20, 20, 100]^T$, simule a população por 100 anos.
    * Plote a população total em escala semilogarítmica e a proporção de indivíduos em cada estágio ao longo do tempo.
    * Calcule a razão $N_{k+1}/N_k$ entre populações totais de anos consecutivos. Para que valor ela converge? O que acontece com as proporções dos estágios?

3. **Autovalor dominante e autovetores.**
    * Calcule o autovalor dominante $\lambda$ de $L$ com `la.eig` e com o método das potências (implementado por você). Interprete: a população cresce ou decresce, e a que taxa anual?
    * Mostre que o autovetor dominante $\mathbf{w}$ (normalizado para somar 1) é a **distribuição estável de estágios** e compare-o com as proporções do item 2.
    * O autovetor dominante à esquerda $\mathbf{v}$ ($\mathbf{v}^TL = \lambda\mathbf{v}^T$, ou seja, o autovetor de $L^T$), normalizado com $v_1 = 1$, é o **valor reprodutivo** de cada estágio. Qual estágio vale mais para o futuro da população? Por que um ovo vale tão pouco?

4. **Sensibilidade e elasticidade.** A sensibilidade de $\lambda$ a uma entrada $a_{ij}$ de $L$ é dada por
    
    $$s_{ij} = \frac{\partial\lambda}{\partial a_{ij}} = \frac{v_i\,w_j}{\mathbf{v}\cdot\mathbf{w}},$$
    
    e a **elasticidade** $e_{ij} = \dfrac{a_{ij}}{\lambda}\,s_{ij}$ mede o efeito de uma mudança **proporcional** em $a_{ij}$.
    * Calcule a matriz de elasticidades, mostre-a como um mapa de calor (`plt.imshow`, com os valores escritos nas células) e verifique que a soma das elasticidades é 1.
    * Confira a fórmula de $s_{ij}$ para uma entrada de sua escolha, calculando a derivada numericamente por diferenças finitas (como na primeira parte do curso).
    * Some as elasticidades das fecundidades (primeira linha), das permanências (diagonal) e das transições (subdiagonal). Quais entradas têm mais influência sobre $\lambda$?

5. **Cenários de manejo.**
    * **Proteção das praias:** suponha que a sobrevivência dos ovos e filhotes chegue a 100% ($a_{21} = 1$). Qual é o novo $\lambda$? Isso basta para salvar a população?
    * **Proteção no mar:** dispositivos de escape instalados nas redes de pesca de camarão (os TEDs, *turtle excluder devices*) aumentam a sobrevivência dos juvenis grandes. Modele isso multiplicando as entradas da coluna 3 abaixo da primeira linha ($a_{33}$ e $a_{43}$) por um fator $f$. Escreva uma função `lam(f)` e encontre, por busca de raízes, o fator $f$ que leva a $\lambda = 1$. Qual é o maior $f$ biologicamente possível? (Lembre que a sobrevivência anual não pode passar de 1.)
    * Repita o cálculo do item anterior para cada estágio $j = 1, \dots, 7$ e apresente em uma tabela o aumento percentual de sobrevivência necessário para atingir $\lambda = 1$, indicando os casos impossíveis.

6. **Investigação.** Com base nos itens 4 e 5, qual política de conservação você recomendaria e por quê? Explique, em termos biológicos, por que proteger os ovos (que parece a estratégia mais intuitiva) tem pouco efeito sobre $\lambda$, e relacione com o valor reprodutivo e as elasticidades. Esse estudo contribuiu para que os TEDs se tornassem obrigatórios na pesca de camarão nos EUA (e, mais tarde, no Brasil).